# Step 3. Detect vertebrae in a folder of radiographs

Runs the trained model on every image in a folder, keeps detections with confidence above 0.7, removes duplicates with non-maximum suppression (IoU 0.3) and saves boxes and corner points as one JSON file per image.

**Input:** model weights and image folder. **Output:** JSON annotations used in step 5.

Code comments are in Spanish.

In [ ]:
from google.colab import drive
# Montar Drive de Google
drive.mount('/content/drive')

In [ ]:
# Definir las rutas
weights_path = 'MODELO_V2.pth' #Colocar ruta del modelo a utilizar para realizar predicciones
image_folder_path = 'IMAGENES_PARA_DETECCION' #Colocar ruta de imagenes a utilizar para realizar predicciones
output_folder_path = 'CARPETA_DE_ANOTACIONES' #Colocar ruta de anotaciones realizadas por el modelo

In [ ]:
# Instalar las bibliotecas necesarias
!pip install torch torchvision
!pip install opencv-python
!pip install numpy
!pip install gdown

# Importar las bibliotecas necesarias
import os
import json
import torch
import torchvision
from torchvision.models.detection import keypointrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import cv2
import numpy as np
from PIL import Image
from torchvision.ops import nms
from torchvision.models.detection.rpn import AnchorGenerator

In [ ]:
# Cargar el Modelo
def get_model(num_keypoints, weights_path):

    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   rpn_anchor_generator=anchor_generator,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 2, # El fondo es la primera clase, el objeto es la segunda
                                                                   pretrained_backbone=True)
    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)

    return model
model = get_model(num_keypoints = 4, weights_path=weights_path)
model.eval()

In [ ]:
# Mover el model a una GPU si hay una dispoible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Iterar sobre todas las imágenes en la carpeta
for image_name in os.listdir(image_folder_path):
    if image_name.endswith(('.jpg', '.png', '.jpeg')):
        image_path = os.path.join(image_folder_path, image_name)

        # Cargar y preprocesar la imagen
        image = Image.open(image_path).convert("RGB")
        image_tensor = F.to_tensor(image).unsqueeze(0).to(device)

        # Realizar la inferencia
        with torch.no_grad():
            prediction = model(image_tensor)

        # Filtrar predicciones según el umbral de confianza y el umbral de IoU
        confidence_threshold = 0.7
        iou_threshold = 0.3

        boxes = prediction[0]['boxes'][prediction[0]['scores'] > confidence_threshold]
        scores = prediction[0]['scores'][prediction[0]['scores'] > confidence_threshold]
        keypoints = prediction[0]['keypoints'][prediction[0]['scores'] > confidence_threshold]

        # Aplicar supresión no máxima utilizando box_iou
        keep = nms(boxes, scores, iou_threshold)

        # Preparar anotaciones en formato Keypoint RCNN
        annotations = {
            'boxes': boxes[keep].cpu().numpy().tolist(),
            'labels': [0] * len(keep),  # Suponiendo solo una clase (0) en el formato Keypoint RCNN
            'keypoints': keypoints[keep].cpu().numpy().tolist(),
        }

        # Guardar anotaciones en un archivo JSON
        output_filename = os.path.join(output_folder_path, f'{os.path.splitext(image_name)[0]}.json')
        with open(output_filename, 'w') as json_file:
            json.dump(annotations, json_file)

print("Inferencia completada.")

Inferencia completada.

In [ ]:
import random

# Obtener una imagen aleatoria de la salida de inferencia
random_image_name = random.choice(os.listdir(output_folder_path))
random_image_path = os.path.join(output_folder_path, random_image_name)

# Cargar las anotaciones de la imagen aleatoria
with open(random_image_path, 'r') as json_file:
    annotations = json.load(json_file)

# Cargar la imagen para visualización
random_image = cv2.imread(os.path.join(image_folder_path, f"{os.path.splitext(random_image_name)[0]}.jpg"))
random_image = cv2.cvtColor(random_image, cv2.COLOR_BGR2RGB)

# Dibujar cajas delimitadoras
for box, keypoints in zip(annotations['boxes'], annotations['keypoints']):
    box = [int(coord) for coord in box]
    cv2.rectangle(random_image, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)

    # Dibujar puntos clave
    for kp in keypoints:
        x, y, v = map(int, kp)
        if v == 1:  # Verificar visibilidad
            cv2.circle(random_image, (x, y), 5, (255, 0, 0), -1)

# Guardar la imagen
output_image_path = 'RUTA_IMAGEN_ALEATORIA' #Colocar ruta para guardar una imagen aleatoria donde se reflejara la inferencia realizada
cv2.imwrite(output_image_path, cv2.cvtColor(random_image, cv2.COLOR_RGB2BGR))

# Proporcionar la ruta de la imagen guardada para inspección manual
print("Ejemplo aleatorio guardado en:", output_image_path)